In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from tqdm import tqdm
from typing import List
tqdm.pandas()
sys.path.append("../../../")
plt.rcParams["font.size"] = 7

from src.data.data_splits import generate_split_mask
from src.data.constants import CXR_SHORT_LABELS, CXR_LABELS

In [ ]:
data_dir = Path("../../../data/raw/mimic-cxr/mimic-cxr-jpg")
assert data_dir.exists() and data_dir.is_dir()

In [ ]:
metadata_df = pd.read_csv(data_dir / 'mimic-cxr-2.0.0-metadata.csv')
diagnosis_df = pd.read_csv(data_dir / 'mimic-cxr-2.0.0-chexpert.csv')
patients_df = pd.read_csv("../../../data/raw/mimiciv/3.1/patients.csv")

In [ ]:
def get_info(df):
    n_images = len(df)
    n_patients = df.subject_id.nunique()
    return n_images, n_patients

In [ ]:
metadata_df

In [ ]:
metadata_df.columns

In [ ]:
patients_df

In [ ]:
df_cxr = pd.merge(metadata_df, diagnosis_df, on=['subject_id', 'study_id'], how='left', validate='many_to_one') # merge with  metadata
df_cxr = pd.merge(df_cxr, patients_df, on='subject_id') # merge with MIMIC-IV patient table to get age and sex information
total_info = get_info(df_cxr)
print(f"Total: {total_info[0]} images, {total_info[1]} patients")

In [ ]:
# drop weird cases where No Finding is not correctly indicated (all zero label vector)
zero_rows = np.all(df_cxr[CXR_LABELS].values == 0, axis=-1)
zero_label_idcs = np.where(zero_rows)[0]
print(" dropping incorrect No Finding labels for %d images" % len(zero_label_idcs))
df_cxr = df_cxr.loc[~zero_rows].reset_index(drop=True)

In [ ]:
#add path column
df_cxr.subject_id = df_cxr.subject_id.astype(str)
df_cxr.study_id = df_cxr.study_id.astype(str)
df_cxr.insert(2, "path", "")
df_cxr.path = df_cxr.subject_id.str[0:2]
df_cxr.path = "p" + df_cxr.path
df_cxr.path = df_cxr.path + "/p" + df_cxr.subject_id + "/s" + df_cxr.study_id + "/" + df_cxr.dicom_id + ".jpg"

In [ ]:
df_cxr.ViewPosition.value_counts()

### Create timestamp column in datetime format

In [ ]:
MIMIC_TIME_FORMATS = {
    "combined_std": "%Y-%m-%d %H%M%S",
    "date_only": "%Y-%m-%d",
    "combined_compact": "%Y%m%d %H%M%S",
}
study_time_processed = (
    df_cxr["StudyTime"]
    .fillna("")
    .astype(str)
    .str.replace(r"\..*", "", regex=True)  # Remove fractional seconds
    .str.zfill(6)
)  # Pad to HHMMSS

datetime_str_combined = (
    df_cxr["StudyDate"].fillna("").astype(str) + " " + study_time_processed
)
timestamp_col = pd.to_datetime(
    datetime_str_combined,
    format=MIMIC_TIME_FORMATS["combined_std"],
    errors="coerce",  # Set errors='coerce' to turn failures into NaT (Not a Time)
)
timestamp_col = timestamp_col.fillna(
    pd.to_datetime(
        df_cxr["StudyDate"],  # Parse only the original StudyDate column
        format=MIMIC_TIME_FORMATS["date_only"],
        errors="coerce",
    )
)
timestamp_col = timestamp_col.fillna(
    pd.to_datetime(
        datetime_str_combined,
        format=MIMIC_TIME_FORMATS["combined_compact"],
        errors="coerce",
    )
)
df_cxr["timestamp"] = timestamp_col

In [ ]:
# calcuate age at acquisition using anchor age
df_cxr = df_cxr.rename(columns={'gender': 'sex'})
df_cxr = df_cxr.rename(columns={'anchor_age': 'age'})
study_year = df_cxr.timestamp.dt.year
delta_years = study_year - df_cxr['anchor_year']
df_cxr['age'] = df_cxr['age'] + delta_years

In [ ]:
df_cxr.age.plot.hist(figsize=(3,2))

In [ ]:
df_cxr.sex.value_counts()

### Map all uncertain labels to 1.0


In [ ]:
df_cxr[CXR_LABELS]

In [ ]:
(df_cxr[CXR_LABELS]==-1.0).sum(axis=0)

In [ ]:
# apply label strategy UNCERTAIN_Zero: replace -1 with 0 (uncertain -> negative)
df_cxr[CXR_LABELS] = df_cxr[CXR_LABELS].replace(-1.0, 0.0)
# replace NaN with zero
df_cxr[CXR_LABELS] = df_cxr[CXR_LABELS].fillna(0.0)

## Data splits

In [ ]:
print(f"Total: {len(df_cxr.study_id.unique())} studies {len(df_cxr.subject_id.unique())} patients, {len(df_cxr)} images")

In [ ]:
all_pids = pd.Series(df_cxr.subject_id.unique())
# patient-wise 80-10-10 train-val-test split
val_test_pids = all_pids.sample(n=15_000, random_state=42, replace=False)
val_pids = val_test_pids.sample(n=5_000, random_state=42, replace=False)
test_pids = val_test_pids[~val_test_pids.isin(val_pids)]
print(f"# val patients: {len(val_pids)}, # test patients: {len(test_pids)}")
df_cxr.loc[:, "study_id"] = pd.to_numeric(df_cxr.study_id)

traindf = df_cxr[~df_cxr.subject_id.isin(val_test_pids)]
val_df = df_cxr[df_cxr.subject_id.isin(val_pids)]
test_df = df_cxr[df_cxr.subject_id.isin(test_pids)]
traindf.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

In [ ]:
from src.data.data_splits import generate_split_mask

In [ ]:
print(f"Train set: {get_info(traindf)}")
print(f"Val set: {get_info(val_df)}")
print(f"Test set: {get_info(test_df)}")

In [ ]:
# some patient have a lot of images ...
import scipy
plt.figure(figsize=(2.5, 1.75))
counts = df_cxr.groupby('subject_id').size().values
mean_count = np.mean(counts)
median_count = np.median(counts)
percentile = np.percentile(counts, 90)
# empirical survival function of counts
res = scipy.stats.ecdf(counts)
res.sf.plot(color="blue")
plt.axvline(mean_count, color='red', label=f"Mean: {mean_count:.2f}")
plt.axvline(median_count, color='green', label=f"Median: {median_count}")
plt.axvline(percentile, color='orange', label=f"90th perc: {percentile}")
plt.xlabel("Number of Images per Patient")
plt.ylabel("1 - Cumulative Probability")
plt.legend()
plt.yscale("log")
plt.title("MIMIC-CXR: #Images per Patient")
plt.savefig("../../../figs/mimic-cxr/mimic_image_stats.pdf", bbox_inches='tight')

In [ ]:
# temporal patient-specific split
eval_mask = generate_split_mask(
    traindf,
    patient_id_col="subject_id",
    timestamp_col="timestamp",
    label_cols=CXR_LABELS,
    n_holdout_classes=1,
)

train_df_historical = traindf[~eval_mask]
train_df_future = traindf[eval_mask]
del traindf
print(f"Train (historical): {get_info(train_df_historical)}")
print(f"Train (future): {get_info(train_df_future)}")
print(f"Validation: {get_info(val_df)}")
print(f"Test: {get_info(test_df)}")

In [ ]:
# Sanity check. Every patient in train_df_future needs to also be present in train_df_historical
eval_pids = pd.Series(train_df_future.subject_id.unique())
assert sum(eval_pids.isin(train_df_historical["subject_id"])) == len(
    eval_pids
), f"Every patient in longitudinal eval set should also be in train set: {sum(eval_pids.isin(train_df_historical['subject_id']))} vs {len(eval_pids)}"

In [ ]:
# we create a helper dataframe that indicates all/any diseases present per patient in the training (historical) set. This is useful for stratification and analysis later on. 
present_diseases_by_patient = train_df_historical.groupby('subject_id')[CXR_LABELS].max()
present_diseases_by_patient

In [ ]:
# we generate two new columns for each future record
    # 'seen_diseases': comma-separated list of disease names that were already present in the historical records of the same patient
    # 'unseen_diseases': comma-separated list of disease names that are present in the future record but were not present in the historical records of the same patient

def case_stratification_by_historical_disease_presence(row:pd.Series, historical_diseases_by_patient:pd.DataFrame, label_cols:List[str], patient_id_col:str="subject_id", verbose:bool=False) -> bool:
    pid = row[patient_id_col]
    label = row[label_cols]
    present_diseases_historical = historical_diseases_by_patient.loc[pid]
    assert present_diseases_historical.shape == label.shape, f"Shape mismatch: {present_diseases_historical.shape} vs {label.shape}"
    positive_unseen = label[(label == 1) & (present_diseases_historical == 0)]
    positive_seen = label[((label == 1) & (present_diseases_historical == 1)) | ((label == 0) & (present_diseases_historical == 0))]
    print(f"Patient {pid} has unseen diseases: {positive_unseen.index.tolist()}") if verbose and len(positive_unseen) > 0 else None
    print(f"Patient {pid} has seen diseases: {positive_seen.index.tolist()}") if verbose and len(positive_seen) > 0 else None
    unseen_labels = positive_unseen.index.tolist() if len(positive_unseen) > 0 else []
    seen_labels = positive_seen.index.tolist() if len(positive_seen) > 0 else []
    assert set(unseen_labels).isdisjoint(set(seen_labels)), f"Seen and unseen labels should be disjoint: {unseen_labels} vs {seen_labels}"
    unseen_labels_str = ",".join(unseen_labels)
    seen_labels_str = ",".join(seen_labels)
    return pd.Series([seen_labels_str, unseen_labels_str]) # must return pd.Series to fill two columns simultaneously

In [ ]:
train_df_future[['seen_diseases', 'unseen_diseases']] = train_df_future.progress_apply(
    lambda row: case_stratification_by_historical_disease_presence(
        row,
        historical_diseases_by_patient=present_diseases_by_patient,
        label_cols=CXR_LABELS,
        patient_id_col="subject_id",
        verbose=False
    ),
    axis=1,
    result_type='expand'
)
train_df_future['has_unseen_disease'] = train_df_future['unseen_diseases'].apply(lambda x: len(x) > 0)
train_df_future['has_seen_disease'] = train_df_future['seen_diseases'].apply(lambda x: len(x) > 0)

In [ ]:
train_df_future["unseen_diseases"].value_counts()

In [ ]:
train_df_future["seen_diseases"].value_counts()

In [ ]:
train_df_future['has_unseen_disease'].value_counts()

In [ ]:
train_df_future["has_seen_disease"].value_counts()

In [ ]:
train_df_historical.to_csv("../../../data/csv/mimic-cxr_train_historical.csv", index=False)
train_df_future.to_csv("../../../data/csv/mimic-cxr_train_future.csv", index=False)
val_df.to_csv("../../../data/csv/mimic-cxr_val.csv", index=False)
test_df.to_csv("../../../data/csv/mimic-cxr_test.csv", index=False)